In [1]:
import matplotlib
matplotlib.rcParams['font.serif'] = ['Times'] + matplotlib.rcParams['font.serif']
matplotlib.rcParams['font.size'] = 6
matplotlib.rcParams['text.usetex'] = False
matplotlib.rcParams["ps.usedistiller"] = 'xpdf'
matplotlib.rcParams['font.family'] = 'serif'
matplotlib.rcParams['font.weight'] = 'normal'
matplotlib.rcParams["mathtext.fontset"] = 'cm'

In [2]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import random
import math

import pandas as pd


import copy

import cvxpy
cp = cvxpy

import figurefirst as fifi

from braid_analysis import braid_analysis_plots

In [3]:
from align_course_direction_analysis import unifying_algo_analysis as uaa
from align_course_direction_analysis import unifying_algo_plots as uap

# Helper Functions

In [4]:
#from unifying_algo_analysis_helper import *
from set_zorder_functions import *

In [5]:
def clean_labels(ax, show_labels, spines=['left', 'bottom']):
    if show_labels:
        ax.set_xticklabels(['', '$0$', '$.68$', '', '$2$', '$3$', '$4$', '$5$'])
    else:
        ax.set_xticklabels([])
        
    ax.set_yticks([-np.pi, -np.pi/2, 0, np.pi/2, np.pi])
    ax.set_ylim(-np.pi, np.pi)
    
    if show_labels:
        ax.set_yticklabels(['$-\pi$', '','$0$','','$\pi$',])
    else:
        ax.set_yticklabels([])
        
    fifi.mpl_functions.adjust_spines(ax, ['left', 'bottom'])
    
    if show_labels:
        ax.set_ylabel('Course direction', labelpad=-2)
        ax.set_xlabel('Time relative to flash (s)')
    else:
        ax.set_ylabel('')
        ax.set_xlabel('')
    
    ax.tick_params(axis='y', pad=2)
    ax.tick_params(axis='x', pad=2)
    
    fifi.mpl_functions.adjust_spines(ax, ['left', 'bottom'],
                                     tick_length=2.5,
                                     spine_locations={'left': 5, 'bottom': 5},
                                     linewidth=0.5)
    fifi.mpl_functions.set_fontsize(ax, 6)

In [6]:
TRANSLATION = True

In [7]:
FIGURE_NAME = 'unifying_analysis.svg'

In [8]:
if TRANSLATION:
    FIGURE_NAME = 'unifying_analysis_translationTrue.svg'

# With real laminar data

In [9]:
show_labels = False

In [10]:
layout = fifi.svg_to_axes.FigureLayout(FIGURE_NAME, autogenlayers=True, make_mplfigures=True, hide_layers=[], dpi=600)
plt.close('all')

In [11]:
braid_df_fname =  '../../../../Data/Experimental_Fly_Data/flies_laminar_c1xwt_preprocessed_optotrigger_trimmed.hdf'
df_laminar = pd.read_hdf(braid_df_fname)

In [12]:
# get a single trajectory
braid_df = df_laminar[df_laminar.intensity==100]
obj_id_key = 'obj_id_unique_event'
obj_id = braid_df[obj_id_key].unique()[40] # << 36 is a good demo, 40 is beautiful, 41 is nice
trajec = braid_df[braid_df[obj_id_key]==obj_id]
trajec = trajec.dropna()

In [13]:
if 0:
    braid_analysis_plots.plot_arrowhead_trajectory(trajec.x.values, trajec.y.values)
else:
    ax = layout.axes[('laminar_example', 'trajec')]

    x_pos = trajec.x.values
    y_pos = trajec.y.values
    braid_analysis_plots.plot_arrowhead_trajectory(x_pos[0:20], y_pos[0:20], color='gray', linewidth=1, ax=ax, arrow_length=0)
    braid_analysis_plots.plot_arrowhead_trajectory(x_pos[20:87], y_pos[20:87], color='red', linewidth=1, ax=ax, arrow_length=0)
    braid_analysis_plots.plot_arrowhead_trajectory(x_pos[87:], y_pos[87:], color='black', linewidth=1, ax=ax, arrow_length=0.03)
    
    
    ax.set_xlim(-0.15, 0.15)
    ax.set_ylim(-0.15, 0.05)
    ax.set_aspect('equal')
    
    fifi.mpl_functions.adjust_spines(ax, [])

In [14]:
course = trajec.course_smoothish.values
abs_min_ix = 120 # 1 second after the flash start; roughly 300 ms after flash end. Adjust based on your data.
abs_max_ix = 500 # 5 seconds after the flash start. 
min_ix_range = 190 # Use a ~2 sec window for fit
max_ix_range = 210 # Use a ~2 sec window for fit

In [15]:
unifying_algo_fit = uaa.bootstrap_miop_affine_fit(course, 
                                                  abs_min_ix, abs_max_ix, min_ix_range, max_ix_range, 
                                                  npoints = 50, 
                                                  n_bootstraps=10, 
                                                  use_cvx_affine=True, 
                                                  include_translation=TRANSLATION)

Set parameter Username
Set parameter LicenseID to value 2753061
Academic license - for non-commercial use only - expires 2026-12-11


In [16]:
# Sort 
sorted_fit = unifying_algo_fit.sort_values('rmse_affine')
best_ix = sorted_fit.index[0]
best_unifying_algo_fit = unifying_algo_fit.iloc[best_ix]

In [17]:
sorted_fit.keys()

Index(['min_ix', 'max_ix', 'abs_min_ix', 'abs_max_ix', 'min_ix_range',
       'max_ix_range', 'slope', 'intercept', 'rotation', 'major_axis',
       'minor_axis', 'AM_0_0', 'AM_0_1', 'AM_1_0', 'AM_1_1', 'tx', 'ty', 'rho',
       'mean_residuals', 'axis_ratio', 'rmse_affine', 'shift_ix', 'iteration',
       'obj_id_unique_event', 'timestep_sec'],
      dtype='object')

In [18]:
ax = layout.axes[('laminar_example', 'course')]
uap.plot_model_fit(course, best_unifying_algo_fit, 
                   flash_frame_start=20,
                   flash_frame_end=88,
                   clean_spines=True,
                   ax=ax,
                   course_marker_size=2,
                   linear_marker_size=2,
                   affine_marker_size=2,
                  )

In [19]:
print('median slope: ', unifying_algo_fit['slope'].abs().median())
print('median axis ratio: ', unifying_algo_fit['axis_ratio'].abs().median())
print('weighted median: ', uaa.get_weighted_value(unifying_algo_fit, col='axis_ratio', use='rmse_affine'))

median slope:  0.051418459172590275
median axis ratio:  0.07993604674567559
weighted median:  0.09797531556017208


In [20]:
clean_labels(ax, show_labels)

In [21]:
set_selective_rasterization(ax, rasterize_markers=['.'], raster_zorder=-1)
set_selective_rasterization(ax, rasterize_collections=[mcollections.PolyCollection], raster_zorder=-2)

layout.append_figure_to_layer(layout.figures['laminar_example'], 'laminar_example', cleartarget=True)
layout.write_svg(FIGURE_NAME)

# With real still air data

In [22]:
show_labels = True

In [23]:
layout = fifi.svg_to_axes.FigureLayout(FIGURE_NAME, autogenlayers=True, make_mplfigures=True, hide_layers=[], dpi=600)
plt.close('all')

In [24]:
braid_df_fname =  '../../../../Data/Experimental_Fly_Data/flies_stillair_c1xwt_preprocessed_optotrigger_trimmed.hdf'
df_still = pd.read_hdf(braid_df_fname)

In [25]:
# get a single trajectory
braid_df = df_still[df_still.intensity==100]
obj_id_key = 'obj_id_unique_event'
# obj_id = braid_df[obj_id_key].unique()[36] << 36 is a good demo, 29 & 35 is a good one that switches direction of circling
obj_id = braid_df[obj_id_key].unique()[36] # '20220817_173045_4200_104' is a tricky one
trajec = braid_df[braid_df[obj_id_key]=='20220817_173045_4200_104']
trajec = trajec.dropna()

In [26]:
if 0:
    braid_analysis_plots.plot_arrowhead_trajectory(trajec.x.values, trajec.y.values)
else:
    ax = layout.axes[('stillair_example', 'trajec')]

    x_pos = trajec.x.values
    y_pos = trajec.y.values
    braid_analysis_plots.plot_arrowhead_trajectory(x_pos[0:20], y_pos[0:20], color='gray', linewidth=1, ax=ax, arrow_length=0)
    braid_analysis_plots.plot_arrowhead_trajectory(x_pos[20:87], y_pos[20:87], color='red', linewidth=1, ax=ax, arrow_length=0)
    braid_analysis_plots.plot_arrowhead_trajectory(x_pos[87:], y_pos[87:], color='black', linewidth=1, ax=ax, arrow_length=0.03)
    
    
    ax.set_xlim(-0.25, 0.35)
    ax.set_ylim(-0.15, 0.15)
    ax.set_aspect('equal')
    
    fifi.mpl_functions.adjust_spines(ax, [])

In [27]:
course = trajec.course_smoothish.values
abs_min_ix = 120 # 1 second after the flash start; roughly 300 ms after flash end. Adjust based on your data.
abs_max_ix = 500 # 5 seconds after the flash start. 
min_ix_range = 190 # Use a ~2 sec window for fit
max_ix_range = 210 # Use a ~2 sec window for fit

In [28]:
unifying_algo_fit = uaa.bootstrap_miop_affine_fit(course, 
                                                  abs_min_ix, abs_max_ix, min_ix_range, max_ix_range, 
                                                  npoints = 50, 
                                                  n_bootstraps=10, 
                                                  use_cvx_affine=True, 
                                                  include_translation=TRANSLATION)

In [29]:
# Sort 
sorted_fit = unifying_algo_fit.sort_values('rmse_affine')
best_ix = sorted_fit.index[0]
best_unifying_algo_fit = unifying_algo_fit.iloc[best_ix]

In [30]:
sorted_fit.keys()

Index(['min_ix', 'max_ix', 'abs_min_ix', 'abs_max_ix', 'min_ix_range',
       'max_ix_range', 'slope', 'intercept', 'rotation', 'major_axis',
       'minor_axis', 'AM_0_0', 'AM_0_1', 'AM_1_0', 'AM_1_1', 'tx', 'ty', 'rho',
       'mean_residuals', 'axis_ratio', 'rmse_affine', 'shift_ix', 'iteration',
       'obj_id_unique_event', 'timestep_sec'],
      dtype='object')

In [31]:
ax = layout.axes[('stillair_example', 'course')]
uap.plot_model_fit(course, best_unifying_algo_fit, 
                   flash_frame_start=20,
                   flash_frame_end=88,
                   clean_spines=True,
                   ax=ax,
                   course_marker_size=2,
                   linear_marker_size=2,
                   affine_marker_size=2,
                  )

In [32]:
print('median slope: ', unifying_algo_fit['slope'].abs().median())
print('median axis ratio: ', unifying_algo_fit['axis_ratio'].abs().median())
print('weighted median: ', uaa.get_weighted_value(unifying_algo_fit, col='axis_ratio', use='rmse_affine'))

median slope:  0.03437868818198363
median axis ratio:  0.02713875546345917
weighted median:  0.16276268164620508


In [33]:
clean_labels(ax, show_labels)

In [34]:
set_selective_rasterization(ax, rasterize_markers=['.'], raster_zorder=-1)
set_selective_rasterization(ax, rasterize_collections=[mcollections.PolyCollection], raster_zorder=-2)

layout.append_figure_to_layer(layout.figures['stillair_example'], 'stillair_example', cleartarget=True)
layout.write_svg(FIGURE_NAME)

# With real unsteady data

In [35]:
show_labels = False

In [36]:
layout = fifi.svg_to_axes.FigureLayout(FIGURE_NAME, autogenlayers=True, make_mplfigures=True, hide_layers=[], dpi=600)
plt.close('all')

In [37]:
braid_df_fname =  '../../../../Data/Experimental_Fly_Data/flies_splitflow_topon_c1xwt_preprocessed_optotrigger_trimmed.hdf'
df_unsteady = pd.read_hdf(braid_df_fname)

In [38]:
# get a single trajectory
braid_df = df_unsteady[df_unsteady.intensity>0]
obj_id_key = 'obj_id_unique_event'
obj_id = braid_df[obj_id_key].unique()[2] # 0
trajec = braid_df[braid_df[obj_id_key]==obj_id]
trajec = trajec.dropna()

In [39]:
if 0:
    braid_analysis_plots.plot_arrowhead_trajectory(trajec.x.values, trajec.y.values)
else:
    ax = layout.axes[('unsteady_example', 'trajec')]

    x_pos = trajec.x.values
    y_pos = trajec.y.values
    braid_analysis_plots.plot_arrowhead_trajectory(x_pos[0:20], y_pos[0:20], color='gray', linewidth=1, ax=ax, arrow_length=0)
    braid_analysis_plots.plot_arrowhead_trajectory(x_pos[20:87], y_pos[20:87], color='red', linewidth=1, ax=ax, arrow_length=0)
    braid_analysis_plots.plot_arrowhead_trajectory(x_pos[87:], y_pos[87:], color='black', linewidth=1, ax=ax, arrow_length=0.03)
    
    
    ax.set_xlim(-0.4, 0.05)
    ax.set_ylim(-0.1, 0.2)
    ax.set_aspect('equal')
    
    fifi.mpl_functions.adjust_spines(ax, [])

In [40]:
course = trajec.course_smoothish.values
abs_min_ix = 120 # 1 second after the flash start; roughly 300 ms after flash end. Adjust based on your data.
abs_max_ix = 500 # 5 seconds after the flash start. 
min_ix_range = 190 # Use a ~2 sec window for fit
max_ix_range = 210 # Use a ~2 sec window for fit

In [41]:
unifying_algo_fit = uaa.bootstrap_miop_affine_fit(course, 
                                                  abs_min_ix, abs_max_ix, min_ix_range, max_ix_range, 
                                                  npoints = 50, 
                                                  n_bootstraps=10, 
                                                  use_cvx_affine=True, 
                                                  include_translation=TRANSLATION)

In [42]:
# Sort 
sorted_fit = unifying_algo_fit.sort_values('rmse_affine')
best_ix = sorted_fit.index[0]
best_unifying_algo_fit = unifying_algo_fit.iloc[best_ix]

In [43]:
sorted_fit.keys()

Index(['min_ix', 'max_ix', 'abs_min_ix', 'abs_max_ix', 'min_ix_range',
       'max_ix_range', 'slope', 'intercept', 'rotation', 'major_axis',
       'minor_axis', 'AM_0_0', 'AM_0_1', 'AM_1_0', 'AM_1_1', 'tx', 'ty', 'rho',
       'mean_residuals', 'axis_ratio', 'rmse_affine', 'shift_ix', 'iteration',
       'obj_id_unique_event', 'timestep_sec'],
      dtype='object')

In [44]:
ax = layout.axes[('unsteady_example', 'course')]
uap.plot_model_fit(course, best_unifying_algo_fit, 
                   flash_frame_start=20,
                   flash_frame_end=88,
                   clean_spines=True,
                   ax=ax,
                   course_marker_size=2,
                   linear_marker_size=2,
                   affine_marker_size=2,
                  )

In [45]:
print('median slope: ', unifying_algo_fit['slope'].abs().median())
print('median axis ratio: ', unifying_algo_fit['axis_ratio'].abs().median())
print('weighted median: ', uaa.get_weighted_value(unifying_algo_fit, col='axis_ratio', use='rmse_affine'))

median slope:  0.05385500158000711
median axis ratio:  0.21614545214332043
weighted median:  0.2205663613170763


In [46]:
clean_labels(ax, show_labels)

In [47]:
set_selective_rasterization(ax, rasterize_markers=['.'], raster_zorder=-1)
set_selective_rasterization(ax, rasterize_collections=[mcollections.PolyCollection], raster_zorder=-2)

layout.append_figure_to_layer(layout.figures['unsteady_example'], 'unsteady_example', cleartarget=True)
layout.write_svg(FIGURE_NAME)

In [48]:
data = diagnose_axis_elements(ax)

COLLECTIONS (scatter, fill_between, etc.)

Collection 0:
  Type: PolyCollection
  Full type: <class 'matplotlib.collections.PolyCollection'>
  Label: _child0
  Current zorder: -2

Collection 1:
  Type: PolyCollection
  Full type: <class 'matplotlib.collections.PolyCollection'>
  Label: _child1
  Current zorder: -2

LINES (plot, with markers and linestyles)

Line 0:
  Label: _child2
  Marker: '.'
  Linestyle: 'None'
  Color: black
  Linewidth: 1.5
  Current zorder: -1

Line 1:
  Label: _child3
  Marker: '.'
  Linestyle: 'None'
  Color: blue
  Linewidth: 1.5
  Current zorder: -1

Line 2:
  Label: _child4
  Marker: '.'
  Linestyle: 'None'
  Color: magenta
  Linewidth: 1.5
  Current zorder: -1

SUMMARY

Total collections: 2

Collection types found:
  PolyCollection: 2

Total lines: 3

Marker types found:
  '.': 3

RASTERIZATION SUGGESTIONS

To rasterize collections, use:
  rasterize_collections=[
      mcollections.PolyCollection,
  ]

To rasterize by marker type, use:
  rasterize_markers=